In [1]:
# ============================================================
# CELDA 1
# Librerías
# ============================================================

import pandas as pd
import numpy as np
import re

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 250)

In [2]:
# ============================================================
# CELDA 2
# Cargar los estudios incluidos
# ============================================================

archivo = "estudios_primarios.csv"

df = pd.read_csv(
    archivo,
    sep=";",
    encoding="utf-8-sig"
)

print("Total de estudios incluidos:", len(df))
print("Total de columnas:", len(df.columns))

display(
    df[
        [
            "ID",
            "Año",
            "Título",
            "Fuente de conocimiento",
            "Tipo de conocimiento",
            "Representación del conocimiento",
            "RQ1"
        ]
    ].head()
)

Total de estudios incluidos: 65
Total de columnas: 45


,ID,Año,Título,Fuente de conocimiento,Tipo de conocimiento,Representación del conocimiento,RQ1
0,1,2024,A Methodology for Integrating Hierarchical VMAP-Data Structures into an Ontology Using Semantically Represented Analyses,"Especificación industrial VMAP, ontologías de dominio, metadatos de simulación, rutinas de procesamiento y requisitos derivados de casos de uso","Conocimiento de dominio, estructural y procedimental","Ontologías, grafos de conocimiento, RDF/RDFS, OWL y relaciones semánticas",Sí
1,2,2020,A Model Driven Approach to Constructing Knowledge Graph from Relational Database,Esquema y datos almacenados en bases de datos relacionales empresariales,"Conocimiento estructurado de dominio; esquemas, entidades, atributos y relaciones",Grafo de conocimiento; esquema y capa de datos; triples entidad--relación--entidad y entidad--clave--valor; XML/XML Schema,Sí
2,3,2026,A model-driven knowledge graph treatment for process-centric cyber threat mitigation,"BPMN, diagramas de flujo de datos (DFD), metodologías de modelado de amenazas como STRIDE y LINDDUN, taxonomías de amenazas/mitigaciones, literatura y conocimiento de stakeholders","Conocimiento de amenazas, mitigaciones, flujos de datos, procesos de negocio, límites de confianza, controles y relaciones semánticas","DSML híbrido BPMN--DFD--Threat Modeling, ontología, RDF, grafos de conocimiento y triplestore",Sí
3,4,2025,"Aligning DevOps and Microservice Architecture: Empirical Mapping, Taxonomy, and RAG-Based Decision Support","84 publicaciones, pares DevOpsMSA extraídos manualmente, taxonomías existentes, evidencia empírica y conceptual","Conocimiento arquitectónico, operacional, de prácticas DevOps, principios MSA, seguridad y gobernanza","Dataset de más de 200 pares DevOps?MSA, vocabulario controlado/Ontology v1.0, taxonomía de cinco categorías, etiquetas de evidencia y citas de procedencia",Sí
4,6,2026,An Empirical Evaluation of Large Language Models Applying Software Architectural Patterns,Requisitos FR/NFR y SRS; material arquitectónico procedente de libros académicos y fuentes web curadas utilizado mediante RAG,"Conocimiento de requisitos, patrones arquitectónicos, principios de diseño y reglas de microservicios","Texto estructurado, documentos SRS, chunks semánticos, embeddings/vector store y material arquitectónico recuperado mediante RAG",Parcial


In [3]:
# ============================================================
# CELDA 3
# Estudios designados directamente para responder RQ1
#
# Se consideran:
# - Principal
# - Sí
#
# Se excluyen de la tabla principal:
# - Parcial
# - Parcial (no seguridad)
# - Principal (sin seguridad)
# - No aborda
# ============================================================

valores_rq1_directos = [
    "Principal",
    "Sí"
]

df_rq1 = df[
    df["RQ1"].isin(valores_rq1_directos)
].copy()

print("Estudios designados directamente a RQ1:", len(df_rq1))

display(
    df_rq1[
        ["ID", "Título", "RQ1"]
    ].sort_values("ID")
)

Estudios designados directamente a RQ1: 39


,ID,Título,RQ1
0,1,A Methodology for Integrating Hierarchical VMAP-Data Structures into an Ontology Using Semantically Represented Analyses,Sí
1,2,A Model Driven Approach to Constructing Knowledge Graph from Relational Database,Sí
2,3,A model-driven knowledge graph treatment for process-centric cyber threat mitigation,Sí
3,4,"Aligning DevOps and Microservice Architecture: Empirical Mapping, Taxonomy, and RAG-Based Decision Support",Sí
5,8,An ontology-based learning approach for automatically classifying security requirements,Sí
6,9,An ontology-based secure design framework for graph-based databases,Sí
7,10,Automated regulatory compliance for AI systems in the security domain: The case of dual-use deployment,Sí
8,11,Understanding and recommending security requirements from problem domain ontology: A cognitive three-layered approach,Sí
9,13,CONSENT: A Software Architecture for Dynamic and Secure Consent Management,Principal
10,14,E-SCORE: A web-based tool for security requirements engineering,Principal


In [4]:
# ============================================================
# CELDA 4
# Correspondencia ID de la matriz -> BibKey de LaTeX
# ============================================================

bibkey_por_id = {

    1:  "Spelten2024",
    2:  "Sun2020",
    3:  "Chis2026",
    4:  "Khadem2025",
    6:  "Hadjichristofi2026",
    8:  "Li2020",
    9:  "Paneque2024",
    10: "Sabaliauskaite2026",
    11: "Kim2020",
    13: "Zoi2026",
    14: "Hnaini2024",
    16: "Yang2023",
    17: "Ghazal2020",
    18: "Vodyaho2022",
    19: "Hosseini2025",
    21: "MarianPasca2025",
    22: "Li2022",
    24: "Tailhardat2024",
    25: "Chaplia2025",
    26: "Rejithkumar2024",
    27: "Sheng2026",
    28: "Jung2023",
    29: "Salem2026",
    32: "Ersoy2025",
    33: "Zeng2025",
    34: "Gemayel2025",
    37: "Boppana2024",
    38: "Awadid2025",
    39: "Mohammed2022",
    41: "Hu2024",
    43: "Xu2025",
    44: "Oliveira2022",
    45: "Tao2024",
    46: "Gijre2024",
    47: "William2025",
    49: "Guan2024",
    50: "Chen2024",
    51: "Zhang2025",
    52: "Su2025",
    53: "Omrani2024",
    55: "Mukherjee2025",
    56: "Rincon2025",
    57: "Yao2023",
    58: "Mascia2025",
    59: "Bacchiega2022",
    60: "Gstur2024",
    61: "Chen2024b",
    62: "Wang2025",
    66: "Bacchiega2024",
    67: "Jin2025",
    68: "Gao2021",
    69: "Qiao2024",
    71: "Chuang2024",
    73: "Yang2022",
    74: "Zhang2023",
    75: "VergaraVargas2024",
    76: "Lakshmanan2025",
    77: "Yamsani2026",
    78: "Negoita2025",
    79: "Zhang2025b",
    81: "Thanh2025",
    83: "Ding2024",
    84: "Sun2025",
    86: "Xu2024",
    87: "Wang2020"
}

df["BibKey"] = df["ID"].map(bibkey_por_id)
df_rq1["BibKey"] = df_rq1["ID"].map(bibkey_por_id)

print("Estudios sin BibKey:", df["BibKey"].isna().sum())

if df["BibKey"].isna().any():
    display(
        df[df["BibKey"].isna()][
            ["ID", "Título"]
        ]
    )
else:
    print("✓ Los 65 estudios tienen correspondencia bibliográfica.")

Estudios sin BibKey: 0
✓ Los 65 estudios tienen correspondencia bibliográfica.


In [5]:
# ============================================================
# CELDA 5
# Taxonomía de FUENTES DE CONOCIMIENTO
#
# Las categorías NO son excluyentes.
# Un estudio puede utilizar varias fuentes.
# ============================================================

reglas_fuentes = {

    "Estándares, normativas y repositorios de seguridad": [
        r"gdpr",
        r"nis2",
        r"ai act",
        r"nato",
        r"common criteria",
        r"isms",
        r"cwe",
        r"cve",
        r"capec",
        r"cpe",
        r"nvd",
        r"nist",
        r"iec 62443",
        r"owasp",
        r"stig",
        r"mitre",
        r"it-grundschutz",
        r"estándar",
        r"regulac",
        r"legislac"
    ],

    "Ontologías, taxonomías y bases de conocimiento previas": [
        r"ontolog",
        r"taxonom",
        r"knowledge base",
        r"\bkb\b",
        r"wordnet",
        r"score",
        r"bigowl"
    ],

    "Documentos, literatura y corpus externos": [
        r"document",
        r"publicacion",
        r"publicaciones",
        r"literatura",
        r"artículo",
        r"articulos",
        r"pdf",
        r"sitios web",
        r"\bweb\b",
        r"corpus",
        r"wikitext",
        r"the pile",
        r"texto no estructurado",
        r"reseñas",
        r"posts",
        r"conocimiento externo"
    ],

    "Conocimiento experto y de stakeholders": [
        r"expert",
        r"stakeholder",
        r"subject matter",
        r"desarrollador",
        r"preferencias.*usuario",
        r"conocimiento previo del dominio"
    ],

    "Artefactos de software y arquitectura": [
        r"arquitect",
        r"bpmn",
        r"dfd",
        r"openapi",
        r"\bapi\b",
        r"requisitos",
        r"modelo",
        r"metamodel",
        r"schema",
        r"esquema",
        r"rdbms",
        r"base de datos",
        r"diagram",
        r"artefact",
        r"módulos",
        r"entidades de red"
    ],

    "Datos operacionales, telemetría e incidentes": [
        r"\blogs\b",
        r"métric",
        r"metric",
        r"trace",
        r"sensor",
        r"histórico de fallos",
        r"fallos",
        r"incidente",
        r"feedback",
        r"time series",
        r"series temporales",
        r"events",
        r"eventos",
        r"mantenimiento",
        r"actividades de usuario",
        r"historial de tests",
        r"historial.*errores"
    ],

    "Datos estructurados y repositorios empresariales": [
        r"datos estructurados",
        r"datos almacenados",
        r"rdbms",
        r"bases de datos relacionales",
        r"postgresql",
        r"schemas y datos",
        r"datos empresariales",
        r"datos clínicos"
    ],

    "Datos multimodales": [
        r"multimodal",
        r"imágenes",
        r"audio",
        r"video",
        r"datos sanitarios multimodales"
    ]
}

In [6]:
# ============================================================
# CELDA 6
# Taxonomía de TIPOS DE CONOCIMIENTO
# ============================================================

reglas_tipos = {

    "Conocimiento de dominio y contextual": [
        r"dominio",
        r"contexto",
        r"contextual",
        r"categorías semánticas",
        r"hechos financieros",
        r"productos",
        r"médico",
        r"clínic",
        r"empresarial"
    ],

    "Conocimiento arquitectónico y estructural": [
        r"arquitect",
        r"estructural",
        r"componentes",
        r"servicios",
        r"interfaces",
        r"topología",
        r"schema",
        r"esquema",
        r"relaciones/tablas"
    ],

    "Conocimiento de seguridad": [
        r"seguridad",
        r"security",
        r"amenaza",
        r"vulnerab",
        r"riesgo",
        r"ataque",
        r"contramedida",
        r"autoriz",
        r"autentic",
        r"privacidad",
        r"consentimiento",
        r"\bcia\b",
        r"\bbola\b",
        r"\bapt\b"
    ],

    "Conocimiento normativo y regulatorio": [
        r"normativ",
        r"regulator",
        r"legal",
        r"cumplimiento",
        r"políticas",
        r"acreditación"
    ],

    "Conocimiento operacional y diagnóstico": [
        r"operacional",
        r"fallo",
        r"diagnóstico",
        r"root cause",
        r"estado operacional",
        r"mantenimiento",
        r"incident",
        r"degradación",
        r"performance",
        r"failure"
    ],

    "Conocimiento causal y de dependencias": [
        r"causal",
        r"dependenc",
        r"interacciones",
        r"propagación",
        r"relaciones.*servicio",
        r"topología temporal"
    ],

    "Conocimiento experto e histórico": [
        r"experiencia histórica",
        r"casos históricos",
        r"históricos",
        r"experto",
        r"feedback"
    ],

    "Conocimiento documental y factual": [
        r"documental",
        r"texto",
        r"evidencia textual",
        r"contenido documental",
        r"notas clínicas",
        r"hechos",
        r"evidencia contextual",
        r"textual dinámico",
        r"conocimiento textual"
    ],

    "Conocimiento semántico y conceptual": [
        r"semántic",
        r"conceptos",
        r"roles semánticos",
        r"relaciones semánticas"
    ]
}

In [7]:
# ============================================================
# CELDA 7
# Taxonomía de REPRESENTACIONES DEL CONOCIMIENTO
# ============================================================

reglas_representaciones = {

    "Ontologías y modelos semánticos": [
        r"ontolog",
        r"description logic",
        r"\bowl\b",
        r"rdf/rdfs",
        r"semantic model",
        r"modelo semántico"
    ],

    "Grafos de conocimiento": [
        r"knowledge graph",
        r"grafo de conocimiento",
        r"knowledge map",
        r"rdf knowledge graph",
        r"triplestore",
        r"neo4j"
    ],

    "Bases de conocimiento": [
        r"knowledge base",
        r"\bkb\b",
        r"\bgpk\b",
        r"\bdsk\b",
        r"case base",
        r"repositorio privado",
        r"knowledge repository"
    ],

    "Reglas, restricciones y patrones": [
        r"reglas",
        r"rules",
        r"restric",
        r"axiomas",
        r"patrones sintácticos",
        r"sentence templates",
        r"refinement rules",
        r"constraints"
    ],

    "Taxonomías y vocabularios controlados": [
        r"taxonom",
        r"vocabulario controlado",
        r"etiquetas de evidencia"
    ],

    "Evidencia documental y fragmentos": [
        r"document",
        r"chunks",
        r"pasajes",
        r"corpus",
        r"secciones semánticas",
        r"textual summaries",
        r"contexto recuperado"
    ],

    "Embeddings, índices y bases vectoriales": [
        r"embedding",
        r"vector db",
        r"vector database",
        r"faiss",
        r"bm25",
        r"índice denso",
        r"dense/vector",
        r"vector library",
        r"dense passage retrieval"
    ],

    "Grafos estructurales, causales u operacionales": [
        r"grafo de microservicios",
        r"causal graph",
        r"service topology graph",
        r"dependency graph",
        r"calling tree",
        r"grafo dirigido",
        r"heterogeneous",
        r"topology graph"
    ],

    "Modelos, metamodelos y datos estructurados": [
        r"metamodel",
        r"modelos conformes",
        r"schema",
        r"\bsql\b",
        r"sparql",
        r"postgresql",
        r"xml",
        r"triples",
        r"matrix",
        r"matriz",
        r"tuples",
        r"result sets"
    ],

    "Prompts y contexto textual": [
        r"prompt",
        r"rag context",
        r"contexto largo",
        r"textual summaries",
        r"structured reasoning evidence"
    ],

    "Representaciones multimodales": [
        r"multimodal",
        r"imágenes",
        r"audio",
        r"video",
        r"\bvlm\b",
        r"blockchain"
    ]
}

In [8]:
# ============================================================
# CELDA 8
# Funciones de clasificación
# ============================================================

def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    return str(texto).lower()


def clasificar_multietiqueta(texto, reglas):
    """
    Devuelve todas las categorías que coinciden con el texto.

    Las categorías son no excluyentes.
    """
    
    texto = normalizar_texto(texto)
    categorias = []
    
    for categoria, patrones in reglas.items():
        
        encontrado = any(
            re.search(
                patron,
                texto,
                flags=re.IGNORECASE
            )
            for patron in patrones
        )
        
        if encontrado:
            categorias.append(categoria)
    
    return categorias


df_rq1["Categorías fuente"] = (
    df_rq1["Fuente de conocimiento"]
    .apply(
        lambda x: clasificar_multietiqueta(
            x,
            reglas_fuentes
        )
    )
)

df_rq1["Categorías tipo"] = (
    df_rq1["Tipo de conocimiento"]
    .apply(
        lambda x: clasificar_multietiqueta(
            x,
            reglas_tipos
        )
    )
)

df_rq1["Categorías representación"] = (
    df_rq1["Representación del conocimiento"]
    .apply(
        lambda x: clasificar_multietiqueta(
            x,
            reglas_representaciones
        )
    )
)

In [9]:
# ============================================================
# CELDA 9
# Control de calidad
# ============================================================

columnas_control = {
    "Fuente": "Categorías fuente",
    "Tipo": "Categorías tipo",
    "Representación": "Categorías representación"
}

for nombre, columna in columnas_control.items():
    
    sin_categoria = df_rq1[
        df_rq1[columna].apply(len) == 0
    ]
    
    print(
        f"{nombre}: "
        f"{len(sin_categoria)} estudios sin categoría"
    )
    
    if len(sin_categoria) > 0:
        display(
            sin_categoria[
                [
                    "ID",
                    "Título",
                    columna
                ]
            ]
        )

Fuente: 0 estudios sin categoría
Tipo: 0 estudios sin categoría
Representación: 0 estudios sin categoría


In [10]:
# ============================================================
# CELDA 10
# Construcción de tablas de resultados
# ============================================================

def construir_tabla_rq1(
    dataframe,
    columna_categorias,
    orden_categorias
):
    
    resultados = []
    
    for categoria in orden_categorias:
        
        seleccion = dataframe[
            dataframe[columna_categorias]
            .apply(
                lambda categorias:
                categoria in categorias
            )
        ].sort_values("ID")
        
        if len(seleccion) == 0:
            continue
        
        claves = seleccion["BibKey"].tolist()
        
        # Citas LaTeX:
        # \cite{Spelten2024,Sun2020,...}
        citas = "\\cite{" + ",".join(claves) + "}"
        
        resultados.append({
            "Categoría": categoria,
            "Estudios designados": citas,
            "Total": seleccion["ID"].nunique()
        })
    
    resultado = pd.DataFrame(resultados)
    
    # Orden descendente por número de estudios
    resultado = (
        resultado
        .sort_values(
            "Total",
            ascending=False
        )
        .reset_index(drop=True)
    )
    
    return resultado

In [11]:
# ============================================================
# CELDA 11
# FUENTES DE CONOCIMIENTO
# ============================================================

tabla_fuentes = construir_tabla_rq1(
    df_rq1,
    "Categorías fuente",
    list(reglas_fuentes.keys())
)

display(tabla_fuentes)

,Categoría,Estudios designados,Total
0,"Documentos, literatura y corpus externos","\cite{Chis2026,Khadem2025,Hnaini2024,Ghazal2020,Ersoy2025,Gemayel2025,Boppana2024,Awadid2025,Xu2025,Gijre2024,Omrani2024,Mukherjee2025,Jin2025,Qiao2024,Chuang2024,VergaraVargas2024,Lakshmanan2025}",17
1,Artefactos de software y arquitectura,"\cite{Spelten2024,Sun2020,Chis2026,Li2020,Paneque2024,Kim2020,Hosseini2025,MarianPasca2025,Awadid2025,Xu2025,Gstur2024,Gao2021,VergaraVargas2024,Thanh2025}",14
2,"Ontologías, taxonomías y bases de conocimiento previas","\cite{Spelten2024,Chis2026,Khadem2025,Li2020,Paneque2024,Hnaini2024,Ghazal2020,Hosseini2025,Guan2024,Chen2024}",10
3,"Estándares, normativas y repositorios de seguridad","\cite{Li2020,Sabaliauskaite2026,Kim2020,Zoi2026,Hnaini2024,Hosseini2025,MarianPasca2025,Jung2023,VergaraVargas2024}",9
4,Conocimiento experto y de stakeholders,"\cite{Chis2026,Kim2020,Zoi2026,Hnaini2024,Hosseini2025,Jung2023,Chen2024,Gstur2024,Sun2025}",9
5,"Datos operacionales, telemetría e incidentes","\cite{MarianPasca2025,Hu2024,Chen2024,Yao2023,Mascia2025,Negoita2025,Zhang2025b,Sun2025}",8
6,Datos estructurados y repositorios empresariales,"\cite{Sun2020,Negoita2025,Thanh2025}",3
7,Datos multimodales,"\cite{Gemayel2025,Su2025,Yao2023}",3


In [12]:
# ============================================================
# CELDA 12
# TIPOS DE CONOCIMIENTO
# ============================================================

tabla_tipos = construir_tabla_rq1(
    df_rq1,
    "Categorías tipo",
    list(reglas_tipos.keys())
)

display(tabla_tipos)

,Categoría,Estudios designados,Total
0,Conocimiento de dominio y contextual,"\cite{Spelten2024,Sun2020,Kim2020,Hnaini2024,Jung2023,Ersoy2025,Gemayel2025,Boppana2024,Xu2025,Su2025,Omrani2024,Jin2025,Qiao2024,Chuang2024,Negoita2025,Thanh2025}",16
1,Conocimiento arquitectónico y estructural,"\cite{Spelten2024,Sun2020,Khadem2025,Paneque2024,Hosseini2025,Awadid2025,Chen2024,Mukherjee2025,Gstur2024,Gao2021,VergaraVargas2024,Zhang2025b,Thanh2025}",13
2,Conocimiento de seguridad,"\cite{Chis2026,Khadem2025,Li2020,Paneque2024,Sabaliauskaite2026,Kim2020,Zoi2026,Hnaini2024,Hosseini2025,MarianPasca2025,Jung2023,VergaraVargas2024}",12
3,Conocimiento documental y factual,"\cite{Ersoy2025,Boppana2024,Xu2025,Gijre2024,Guan2024,Omrani2024,Mukherjee2025,Qiao2024,Chuang2024,Lakshmanan2025,Negoita2025,Thanh2025}",12
4,Conocimiento operacional y diagnóstico,"\cite{Khadem2025,Hu2024,Chen2024,Yao2023,Mascia2025,Gao2021,Zhang2025b,Sun2025}",8
5,Conocimiento semántico y conceptual,"\cite{Chis2026,Li2020,Ghazal2020,Boppana2024,Gijre2024,Mukherjee2025,Jin2025}",7
6,Conocimiento normativo y regulatorio,"\cite{Sabaliauskaite2026,Zoi2026,Ghazal2020,Ersoy2025,Su2025}",5
7,Conocimiento causal y de dependencias,"\cite{Hu2024,Chen2024,Mascia2025,Zhang2025b}",4
8,Conocimiento experto e histórico,"\cite{MarianPasca2025,Yao2023}",2


In [13]:
# ============================================================
# CELDA 13
# REPRESENTACIONES DEL CONOCIMIENTO
# ============================================================

tabla_representaciones = construir_tabla_rq1(
    df_rq1,
    "Categorías representación",
    list(reglas_representaciones.keys())
)

display(tabla_representaciones)

,Categoría,Estudios designados,Total
0,"Embeddings, índices y bases vectoriales","\cite{MarianPasca2025,Ersoy2025,Gemayel2025,Boppana2024,Awadid2025,Xu2025,Gijre2024,Guan2024,Omrani2024,Mukherjee2025,Yao2023,Jin2025,Qiao2024,Chuang2024,Lakshmanan2025,Negoita2025}",16
1,Evidencia documental y fragmentos,"\cite{Zoi2026,Ersoy2025,Gemayel2025,Awadid2025,Xu2025,Guan2024,Omrani2024,Mukherjee2025,Mascia2025,Qiao2024,Chuang2024,Lakshmanan2025,Negoita2025,Sun2025}",14
2,Ontologías y modelos semánticos,"\cite{Spelten2024,Chis2026,Khadem2025,Li2020,Paneque2024,Sabaliauskaite2026,Kim2020,Zoi2026,Hnaini2024,Ghazal2020,Hosseini2025,Jung2023,Zhang2025b}",13
3,Grafos de conocimiento,"\cite{Sun2020,Chis2026,Paneque2024,Sabaliauskaite2026,Hosseini2025,Hu2024,Xu2025,Gijre2024,Yao2023,Gao2021,VergaraVargas2024,Zhang2025b}",12
4,"Modelos, metamodelos y datos estructurados","\cite{Sun2020,Chis2026,Paneque2024,Hnaini2024,Xu2025,Gstur2024,Negoita2025,Thanh2025,Sun2025}",9
5,Bases de conocimiento,"\cite{Kim2020,Jung2023,Chen2024,Jin2025,Chuang2024}",5
6,"Reglas, restricciones y patrones","\cite{Li2020,Paneque2024,Sabaliauskaite2026,Hosseini2025,Gstur2024}",5
7,Representaciones multimodales,"\cite{Zoi2026,Awadid2025,Su2025,Yao2023,Jin2025}",5
8,Prompts y contexto textual,"\cite{Zoi2026,MarianPasca2025,Mukherjee2025,Sun2025}",4
9,"Grafos estructurales, causales u operacionales","\cite{Chen2024,Mascia2025,Sun2025}",3


In [14]:
# ============================================================
# CELDA 14
# Escapar caracteres reservados de LaTeX
# ============================================================

def escapar_latex(texto):
    
    texto = str(texto)
    
    reemplazos = {
        "&": r"\&",
        "%": r"\%",
        "#": r"\#",
        "_": r"\_"
    }
    
    for original, reemplazo in reemplazos.items():
        texto = texto.replace(
            original,
            reemplazo
        )
    
    return texto

In [15]:
# ============================================================
# CELDA 15
# Generar tabla en formato LaTeX / MDPI
# ============================================================

def generar_tabla_latex(
    tabla,
    titulo,
    etiqueta,
    encabezado_primera_columna
):
    
    lineas = []
    
    lineas.append(r"\begin{table}[H]")
    lineas.append(
        rf"\caption{{{titulo}.\label{{{etiqueta}}}}}"
    )
    
    lineas.append(r"%\isPreprints{\centering}{%")
    lineas.append(
        r"\begin{adjustwidth}{-\extralength}{0cm}"
    )
    lineas.append(r"%}")
    
    lineas.append(
        r"%\isPreprints{\begin{tabularx}{\textwidth}{CCC}}{%"
    )
    
    # Primera columna: categoría
    # Segunda: citas
    # Tercera: total
    lineas.append(
        r"\begin{tabularx}{\fulllength}"
        r"{p{0.30\fulllength} X p{0.08\fulllength}}"
    )
    
    lineas.append(r"%}")
    lineas.append(r"\toprule")
    
    lineas.append(
        rf"\textbf{{{encabezado_primera_columna}}} & "
        r"\textbf{Estudios designados} & "
        r"\textbf{Total} \\"
    )
    
    lineas.append(r"\midrule")
    lineas.append("")
    
    for _, fila in tabla.iterrows():
        
        categoria = escapar_latex(
            fila["Categoría"]
        )
        
        # No escapar las citas:
        # deben conservar \cite{...}
        citas = fila["Estudios designados"]
        
        total = fila["Total"]
        
        lineas.append(
            f"{categoria} & "
            f"{citas} & "
            f"{total} \\\\"
        )
        
        lineas.append("")
    
    lineas.append(r"\bottomrule")
    lineas.append(r"\end{tabularx}")
    
    lineas.append(r"%\isPreprints{}{%")
    lineas.append(r"\end{adjustwidth}")
    lineas.append(r"%}")
    lineas.append(r"\end{table}")
    
    return "\n".join(lineas)

In [16]:
# ============================================================
# CELDA 16
# Tabla LaTeX: fuentes de conocimiento
# ============================================================

latex_fuentes = generar_tabla_latex(
    tabla_fuentes,
    titulo=(
        "Fuentes de conocimiento identificadas "
        "en los estudios que responden a RQ1"
    ),
    etiqueta="tab:rq1-fuentes",
    encabezado_primera_columna="Fuente de conocimiento"
)

print(latex_fuentes)

\begin{table}[H]
\caption{Fuentes de conocimiento identificadas en los estudios que responden a RQ1.\label{tab:rq1-fuentes}}
%\isPreprints{\centering}{%
\begin{adjustwidth}{-\extralength}{0cm}
%}
%\isPreprints{\begin{tabularx}{\textwidth}{CCC}}{%
\begin{tabularx}{\fulllength}{p{0.30\fulllength} X p{0.08\fulllength}}
%}
\toprule
\textbf{Fuente de conocimiento} & \textbf{Estudios designados} & \textbf{Total} \\
\midrule

Documentos, literatura y corpus externos & \cite{Chis2026,Khadem2025,Hnaini2024,Ghazal2020,Ersoy2025,Gemayel2025,Boppana2024,Awadid2025,Xu2025,Gijre2024,Omrani2024,Mukherjee2025,Jin2025,Qiao2024,Chuang2024,VergaraVargas2024,Lakshmanan2025} & 17 \\

Artefactos de software y arquitectura & \cite{Spelten2024,Sun2020,Chis2026,Li2020,Paneque2024,Kim2020,Hosseini2025,MarianPasca2025,Awadid2025,Xu2025,Gstur2024,Gao2021,VergaraVargas2024,Thanh2025} & 14 \\

Ontologías, taxonomías y bases de conocimiento previas & \cite{Spelten2024,Chis2026,Khadem2025,Li2020,Paneque2024,Hnaini202

In [17]:
# ============================================================
# CELDA 17
# Tabla LaTeX: tipos de conocimiento
# ============================================================

latex_tipos = generar_tabla_latex(
    tabla_tipos,
    titulo=(
        "Tipos de conocimiento identificados "
        "en los estudios que responden a RQ1"
    ),
    etiqueta="tab:rq1-tipos",
    encabezado_primera_columna="Tipo de conocimiento"
)

print(latex_tipos)

\begin{table}[H]
\caption{Tipos de conocimiento identificados en los estudios que responden a RQ1.\label{tab:rq1-tipos}}
%\isPreprints{\centering}{%
\begin{adjustwidth}{-\extralength}{0cm}
%}
%\isPreprints{\begin{tabularx}{\textwidth}{CCC}}{%
\begin{tabularx}{\fulllength}{p{0.30\fulllength} X p{0.08\fulllength}}
%}
\toprule
\textbf{Tipo de conocimiento} & \textbf{Estudios designados} & \textbf{Total} \\
\midrule

Conocimiento de dominio y contextual & \cite{Spelten2024,Sun2020,Kim2020,Hnaini2024,Jung2023,Ersoy2025,Gemayel2025,Boppana2024,Xu2025,Su2025,Omrani2024,Jin2025,Qiao2024,Chuang2024,Negoita2025,Thanh2025} & 16 \\

Conocimiento arquitectónico y estructural & \cite{Spelten2024,Sun2020,Khadem2025,Paneque2024,Hosseini2025,Awadid2025,Chen2024,Mukherjee2025,Gstur2024,Gao2021,VergaraVargas2024,Zhang2025b,Thanh2025} & 13 \\

Conocimiento de seguridad & \cite{Chis2026,Khadem2025,Li2020,Paneque2024,Sabaliauskaite2026,Kim2020,Zoi2026,Hnaini2024,Hosseini2025,MarianPasca2025,Jung2023,Vergara

In [18]:
# ============================================================
# CELDA 18
# Tabla LaTeX: representaciones del conocimiento
# ============================================================

latex_representaciones = generar_tabla_latex(
    tabla_representaciones,
    titulo=(
        "Representaciones del conocimiento identificadas "
        "en los estudios que responden a RQ1"
    ),
    etiqueta="tab:rq1-representaciones",
    encabezado_primera_columna=(
        "Representación del conocimiento"
    )
)

print(latex_representaciones)

\begin{table}[H]
\caption{Representaciones del conocimiento identificadas en los estudios que responden a RQ1.\label{tab:rq1-representaciones}}
%\isPreprints{\centering}{%
\begin{adjustwidth}{-\extralength}{0cm}
%}
%\isPreprints{\begin{tabularx}{\textwidth}{CCC}}{%
\begin{tabularx}{\fulllength}{p{0.30\fulllength} X p{0.08\fulllength}}
%}
\toprule
\textbf{Representación del conocimiento} & \textbf{Estudios designados} & \textbf{Total} \\
\midrule

Embeddings, índices y bases vectoriales & \cite{MarianPasca2025,Ersoy2025,Gemayel2025,Boppana2024,Awadid2025,Xu2025,Gijre2024,Guan2024,Omrani2024,Mukherjee2025,Yao2023,Jin2025,Qiao2024,Chuang2024,Lakshmanan2025,Negoita2025} & 16 \\

Evidencia documental y fragmentos & \cite{Zoi2026,Ersoy2025,Gemayel2025,Awadid2025,Xu2025,Guan2024,Omrani2024,Mukherjee2025,Mascia2025,Qiao2024,Chuang2024,Lakshmanan2025,Negoita2025,Sun2025} & 14 \\

Ontologías y modelos semánticos & \cite{Spelten2024,Chis2026,Khadem2025,Li2020,Paneque2024,Sabaliauskaite2026,Kim202

In [19]:
# ============================================================
# CELDA 19
# Guardar las tablas como archivos .tex
# ============================================================

with open(
    "tabla_rq1_fuentes.tex",
    "w",
    encoding="utf-8"
) as archivo:
    archivo.write(latex_fuentes)


with open(
    "tabla_rq1_tipos.tex",
    "w",
    encoding="utf-8"
) as archivo:
    archivo.write(latex_tipos)


with open(
    "tabla_rq1_representaciones.tex",
    "w",
    encoding="utf-8"
) as archivo:
    archivo.write(latex_representaciones)


print("Archivos generados:")
print(" - tabla_rq1_fuentes.tex")
print(" - tabla_rq1_tipos.tex")
print(" - tabla_rq1_representaciones.tex")

Archivos generados:
 - tabla_rq1_fuentes.tex
 - tabla_rq1_tipos.tex
 - tabla_rq1_representaciones.tex


In [20]:
# ============================================================
# CELDA 20
# Matriz de trazabilidad de RQ1
# ============================================================

trazabilidad_rq1 = df_rq1[
    [
        "ID",
        "BibKey",
        "Título",
        "Fuente de conocimiento",
        "Categorías fuente",
        "Tipo de conocimiento",
        "Categorías tipo",
        "Representación del conocimiento",
        "Categorías representación"
    ]
].copy()

trazabilidad_rq1.to_excel(
    "trazabilidad_RQ1.xlsx",
    index=False
)

display(trazabilidad_rq1)

print(
    "Archivo generado: trazabilidad_RQ1.xlsx"
)

,ID,BibKey,Título,Fuente de conocimiento,Categorías fuente,Tipo de conocimiento,Categorías tipo,Representación del conocimiento,Categorías representación
0,1,Spelten2024,A Methodology for Integrating Hierarchical VMAP-Data Structures into an Ontology Using Semantically Represented Analyses,"Especificación industrial VMAP, ontologías de dominio, metadatos de simulación, rutinas de procesamiento y requisitos derivados de casos de uso","[Ontologías, taxonomías y bases de conocimiento previas, Artefactos de software y arquitectura]","Conocimiento de dominio, estructural y procedimental","[Conocimiento de dominio y contextual, Conocimiento arquitectónico y estructural]","Ontologías, grafos de conocimiento, RDF/RDFS, OWL y relaciones semánticas",[Ontologías y modelos semánticos]
1,2,Sun2020,A Model Driven Approach to Constructing Knowledge Graph from Relational Database,Esquema y datos almacenados en bases de datos relacionales empresariales,"[Artefactos de software y arquitectura, Datos estructurados y repositorios empresariales]","Conocimiento estructurado de dominio; esquemas, entidades, atributos y relaciones","[Conocimiento de dominio y contextual, Conocimiento arquitectónico y estructural]",Grafo de conocimiento; esquema y capa de datos; triples entidad--relación--entidad y entidad--clave--valor; XML/XML Schema,"[Grafos de conocimiento, Modelos, metamodelos y datos estructurados]"
2,3,Chis2026,A model-driven knowledge graph treatment for process-centric cyber threat mitigation,"BPMN, diagramas de flujo de datos (DFD), metodologías de modelado de amenazas como STRIDE y LINDDUN, taxonomías de amenazas/mitigaciones, literatura y conocimiento de stakeholders","[Ontologías, taxonomías y bases de conocimiento previas, Documentos, literatura y corpus externos, Conocimiento experto y de stakeholders, Artefactos de software y arquitectura]","Conocimiento de amenazas, mitigaciones, flujos de datos, procesos de negocio, límites de confianza, controles y relaciones semánticas","[Conocimiento de seguridad, Conocimiento semántico y conceptual]","DSML híbrido BPMN--DFD--Threat Modeling, ontología, RDF, grafos de conocimiento y triplestore","[Ontologías y modelos semánticos, Grafos de conocimiento, Modelos, metamodelos y datos estructurados]"
3,4,Khadem2025,"Aligning DevOps and Microservice Architecture: Empirical Mapping, Taxonomy, and RAG-Based Decision Support","84 publicaciones, pares DevOpsMSA extraídos manualmente, taxonomías existentes, evidencia empírica y conceptual","[Ontologías, taxonomías y bases de conocimiento previas, Documentos, literatura y corpus externos]","Conocimiento arquitectónico, operacional, de prácticas DevOps, principios MSA, seguridad y gobernanza","[Conocimiento arquitectónico y estructural, Conocimiento de seguridad, Conocimiento operacional y diagnóstico]","Dataset de más de 200 pares DevOps?MSA, vocabulario controlado/Ontology v1.0, taxonomía de cinco categorías, etiquetas de evidencia y citas de procedencia","[Ontologías y modelos semánticos, Taxonomías y vocabularios controlados]"
5,8,Li2020,An ontology-based learning approach for automatically classifying security requirements,Ontologías y taxonomías previas de requisitos de seguridad; estándares de seguridad; IT-Grundschutz; conocimiento lingüístico; WordNet,"[Estándares, normativas y repositorios de seguridad, Ontologías, taxonomías y bases de conocimiento previas, Artefactos de software y arquitectura]","Activos, propiedades de seguridad, amenazas, contramedidas, mecanismos de seguridad, restricciones de seguridad y relaciones semánticas entre estos conceptos","[Conocimiento de seguridad, Conocimiento semántico y conceptual]","Ontología formal en Description Logic (DL), compatible/reutilizable con OWL; reglas lingüísticas; palabras clave; patrones sintácticos","[Ontologías y modelos semánticos, Reglas, restricciones y patrones]"
6,9,Paneque2024,An ontology-based secure design framework for graph-based databases,Ontología BIGOWL; metamodelos estructurale

Archivo generado: trazabilidad_RQ1.xlsx
